In [8]:
import numpy as np
import pandas as pd
from typing import List, Dict

import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.8.0+cu128
CUDA available: False


In [9]:
MODEL_NAMES = [
    "intfloat/multilingual-e5-small",
    "intfloat/multilingual-e5-base",
    # "Alibaba-NLP/gte-multilingual-base",
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cpu


In [10]:
cte_df = pd.read_csv("../../data/prep/CTE.csv")

In [11]:
cte_df

,Unnamed: 0,CTE_id,CTE_name,category,manufacturer,characteristics
0,0,34863000,Мусорные пакеты 35л,Пакеты полимерные,ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ СПРИН...,"Количество в упаковке:30.00000;Толщина, мкм:50..."
1,1,20647780,Пакеты для мусора,Пакеты полимерные,ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ СПРИН...,Цвет:черный;Ширина:470.00000;Количество рулоно...
2,2,34865337,Мусорные пакеты 240л,Пакеты полимерные,ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ СПРИН...,Количество в упаковке:10.00000;Цвет:черный;Тол...
3,3,38669978,"Радиостанция носимая аналоговая Шеврон, ""АЙТИ-...",Радиостанции цифровые,"ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ""АЙТИ...",Максимальная рабочая температура:50.00000;Коли...
4,4,38515747,Набор реагентов для иммуноферментного определе...,Наборы реактивов/реагентов,"ООО ""ХЕМА""",Концентрация аналита в наибольшей калибровочно...
...,...,...,...,...,...,...
244479,244487,39163435,Гофра для раковины EWRIKA 85001 85036-750 40 м...,Удлинители гофрированные для оборудования сант...,EWRIKA,Ширина:65.00000;Комплектация:Комплект прокладо...
244480,244488,39163497,"Творог Б.Ю. Александров 9%, 180г",Творог,Б.Ю. АЛЕКСАНДРОВ,Наличие вкусовых компонентов:Нет;Вид молочного...
244481,244489,39165297,"Труба VALFEX OPTIMA ф110, с раструбом, L=0.25 ...",Детали и фитинги трубопроводов из полимеров,VALFEX OPTIMA,Материал трубы:полипропилен;Вид товара:Труба;Д...
244482,244490,1374499,Мука пшеничная Макфа весовая,Мука пшеничная хлебопекарная,"АО ""Макфа""",Назначение:пшеничная хлебопекарная;Тарная упак...


In [12]:
# Build canonical text for each CTE item

def build_item_text(row: pd.Series) -> str:
    parts = []
    if pd.notna(row.get("CTE_name")):
        parts.append(f"Наименование: {row['CTE_name']}")
    if pd.notna(row.get("category")):
        parts.append(f"Категория: {row['category']}")
    if pd.notna(row.get("manufacturer")):
        parts.append(f"Производитель: {row['manufacturer']}")
    if pd.notna(row.get("characteristics")):
        parts.append(f"Характеристики: {row['characteristics']}")
    return ". ".join(parts)

cte_sample = cte_df.head(100).copy()
cte_sample["text"] = cte_sample.apply(build_item_text, axis=1)

cte_sample[["CTE_id", "text"]].head()

,CTE_id,text
0,34863000,Наименование: Мусорные пакеты 35л. Категория: ...
1,20647780,Наименование: Пакеты для мусора. Категория: Па...
2,34865337,Наименование: Мусорные пакеты 240л. Категория:...
3,38669978,Наименование: Радиостанция носимая аналоговая ...
4,38515747,Наименование: Набор реагентов для иммунофермен...


In [13]:
# FAISS helpers and encoding utilities

import faiss
from functools import lru_cache

@lru_cache(maxsize=None)
def load_model(model_name: str) -> SentenceTransformer:
    print(f"\n=== Loading model: {model_name} ===")
    model = SentenceTransformer(model_name, device=DEVICE)
    return model


def prepare_texts_for_model(model_name: str, texts, is_query: bool):
    name = model_name.lower()
    if "e5" in name:
        prefix = "query: " if is_query else "passage: "
        return [prefix + t for t in texts]
    return list(texts)


def encode_texts(model_name: str, texts, is_query: bool) -> np.ndarray:
    model = load_model(model_name)
    prepared = prepare_texts_for_model(model_name, texts, is_query=is_query)
    with torch.no_grad():
        emb = model.encode(
            prepared,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
        )
    faiss.normalize_L2(emb)
    return emb.astype("float32")

In [14]:
# Build embeddings and FAISS index for each model

indices: Dict[str, faiss.Index] = {}
embeddings: Dict[str, np.ndarray] = {}
cte_ids_by_model: Dict[str, np.ndarray] = {}

texts = cte_sample["text"].tolist()
cte_ids = cte_sample["CTE_id"].to_numpy()

for model_name in MODEL_NAMES:
    print(f"\n=== Building embeddings and index for: {model_name} ===")
    X = encode_texts(model_name, texts, is_query=False)
    embeddings[model_name] = X
    cte_ids_by_model[model_name] = cte_ids.copy()

    d = X.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(X)
    print("Index size:", index.ntotal)

    indices[model_name] = index


=== Building embeddings and index for: intfloat/multilingual-e5-small ===

=== Loading model: intfloat/multilingual-e5-small ===
Index size: 100

=== Building embeddings and index for: intfloat/multilingual-e5-base ===

=== Loading model: intfloat/multilingual-e5-base ===
Index size: 100


In [15]:
# Search helper

def search_similar_items(model_name: str, query: str, top_k: int = 5):
    if model_name not in indices:
        raise ValueError(f"Unknown model: {model_name}")
    index = indices[model_name]

    q_emb = encode_texts(model_name, [query], is_query=True)

    sims, idx = index.search(q_emb, top_k)
    sims = sims[0]
    idx = idx[0]

    print(f"\n=== Model: {model_name} ===")
    print(f"Query: {query}\n")
    for rank, (i, s) in enumerate(zip(idx, sims), start=1):
        row = cte_sample.iloc[i]
        print(f"{rank:2d}. CTE_id={row['CTE_id']}, sim={s:.3f}")
        print(f"   name: {row['CTE_name']}")
        print(f"   category: {row['category']}")
        print(f"   manufacturer: {row['manufacturer']}")
        print(f"   characteristics: {row['characteristics']}")
        print()

In [16]:
# Example test queries

test_queries = [
    "пакеты для мусора, 30 штук, объем 35 л, черные",
    "полимерные пакеты для мусора, объем 35 литров, ПВД",
]

for q in test_queries:
    for m in MODEL_NAMES:
        search_similar_items(m, q, top_k=5)


=== Model: intfloat/multilingual-e5-small ===
Query: пакеты для мусора, 30 штук, объем 35 л, черные

 1. CTE_id=34863000, sim=0.900
   name: Мусорные пакеты 35л
   category: Пакеты полимерные
   manufacturer: ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ СПРИНТ-ПЛАСТ
   characteristics: Количество в упаковке:30.00000;Толщина, мкм:50.00000;Объем:35.00000;Цвет:черный;Вид материала:ПВД

 2. CTE_id=20647780, sim=0.868
   name: Пакеты для мусора
   category: Пакеты полимерные
   manufacturer: ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ СПРИНТ-ПЛАСТ
   characteristics: Цвет:черный;Ширина:470.00000;Количество рулонов в транспортной коробке:60;Плотность:20.00000;Количество:30;Объем:30.00000;Длина:0.00000;Страна производства:Российская Федерация

 3. CTE_id=34865337, sim=0.855
   name: Мусорные пакеты 240л
   category: Пакеты полимерные
   manufacturer: ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ СПРИНТ-ПЛАСТ
   characteristics: Количество в упаковке:10.00000;Цвет:черный;Толщина, мкм:100.00000;Вид матери